# Prueba CLIP v1 — Fine-tuning con imagenes satelitales PASTIS-R

**Equipo 48** | Febrero 2026

Este notebook realiza fine-tuning de un modelo CLIP (ViT-B/32) pre-entrenado
usando los pares imagen+texto generados en `train_data_v1/`:
- **Imagenes:** composiciones RGB 128x128 (mediana temporal de Sentinel-2)
- **Textos:** captions describiendo los cultivos presentes en cada imagen

El objetivo es que CLIP aprenda a asociar imagenes satelitales con
descripciones de cultivos agricolas para el paper *"Bee my eyes"*.

---
## 1. Setup

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image

import open_clip

plt.style.use('seaborn-v0_8-whitegrid')

# Dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ============================================================
# RUTAS Y CONFIGURACION
# ============================================================

DATA_DIR = Path("./train_data_v1")
METADATA_PATH = Path("../PASTIS-R/metadata.geojson")

# Cargar folds oficiales de PASTIS (evita fuga geografica)
with open(METADATA_PATH) as f:
    geojson = json.load(f)

patch_fold = {}
for feat in geojson["features"]:
    pid = int(feat["properties"]["ID_PATCH"])
    fold = int(feat["properties"]["Fold"])
    patch_fold[pid] = fold

# Solo patches que existen en train_data_v1
available_ids = sorted([
    int(p.stem) for p in DATA_DIR.glob("*.png")
    if (DATA_DIR / f"{p.stem}.txt").exists()
])

# Split: Folds 1-4 = train, Fold 5 = val
train_ids = [pid for pid in available_ids if patch_fold.get(pid, 0) in [1, 2, 3, 4]]
val_ids   = [pid for pid in available_ids if patch_fold.get(pid, 0) == 5]

print(f"Patches disponibles: {len(available_ids):,}")
print(f"Train (folds 1-4):   {len(train_ids):,}")
print(f"Val   (fold 5):      {len(val_ids):,}")

---
## 2. Dataset y DataLoaders

In [ ]:
class SatelliteCLIPDataset(Dataset):
    """Dataset de pares (imagen satelital, caption de cultivos) para CLIP."""

    def __init__(self, data_dir, patch_ids, preprocess, tokenizer):
        self.data_dir = Path(data_dir)
        self.patch_ids = patch_ids
        self.preprocess = preprocess
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, idx):
        pid = self.patch_ids[idx]

        # Imagen: cargar PNG y aplicar preprocess de CLIP (resize 224, normalize)
        img = Image.open(self.data_dir / f"{pid}.png").convert("RGB")
        img_tensor = self.preprocess(img)

        # Texto: leer caption y tokenizar
        caption = (self.data_dir / f"{pid}.txt").read_text(encoding="utf-8").strip()
        text_tokens = self.tokenizer([caption])[0]  # (77,)

        return img_tensor, text_tokens


print("Dataset definido")

---
## 3. Modelo CLIP

Se usa **ViT-B/32** pre-entrenado en LAION-2B: es el backbone mas ligero de CLIP,
rapido de fine-tunear, y suficiente para un primer experimento con ~2,400 pares.

In [ ]:
# Cargar modelo pre-entrenado y sus transforms
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
tokenizer = open_clip.get_tokenizer("ViT-B-32")

model = model.to(device)

# Contar parametros
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parametros totales:     {total_params:,}")
print(f"Parametros entrenables: {train_params:,}")

In [ ]:
# ============================================================
# DATALOADERS
# ============================================================

BATCH_SIZE = 64

train_ds = SatelliteCLIPDataset(DATA_DIR, train_ids, preprocess_train, tokenizer)
val_ds   = SatelliteCLIPDataset(DATA_DIR, val_ids,   preprocess_val,   tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=(device == "cuda"))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(device == "cuda"))

# Verificar un batch
images, texts = next(iter(train_loader))
print(f"Batch de imagenes: {images.shape}")   # (B, 3, 224, 224)
print(f"Batch de tokens:   {texts.shape}")    # (B, 77)

---
## 4. Entrenamiento

**Hiperparametros:**
- **Optimizer:** AdamW con lr=1e-5 (tasa baja para no destruir features pre-entrenadas)
- **Scheduler:** Cosine annealing con warmup lineal de 1 epoca
- **Loss:** InfoNCE simetrica (contrastive loss estandar de CLIP)
- **Epochs:** 20 con early stopping si val_loss no mejora en 5 epocas

In [ ]:
# ============================================================
# CONFIGURACION DE ENTRENAMIENTO
# ============================================================

N_EPOCHS = 20
LR = 1e-5
PATIENCE = 5  # early stopping

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

# Cosine annealing con warmup lineal de 1 epoca
warmup_steps = len(train_loader)  # 1 epoca de warmup
total_steps = N_EPOCHS * len(train_loader)

def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps  # warmup lineal
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))  # cosine decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f"Steps por epoca: {len(train_loader)}")
print(f"Total steps:     {total_steps}")
print(f"Warmup steps:    {warmup_steps}")

In [ ]:
def clip_loss(model, images, texts):
    """Calcula la loss contrastiva simetrica (InfoNCE) de CLIP."""
    image_features = model.encode_image(images)
    text_features = model.encode_text(texts)

    # Normalizar a esfera unitaria
    image_features = F.normalize(image_features, dim=-1)
    text_features = F.normalize(text_features, dim=-1)

    # Similitud escalada
    logit_scale = model.logit_scale.exp()
    logits_per_image = logit_scale * image_features @ text_features.t()
    logits_per_text = logits_per_image.t()

    # Labels: cada imagen i debe matchear con texto i (diagonal)
    labels = torch.arange(len(images), device=images.device)

    # Loss simetrica
    loss_i2t = F.cross_entropy(logits_per_image, labels)
    loss_t2i = F.cross_entropy(logits_per_text, labels)
    loss = (loss_i2t + loss_t2i) / 2

    return loss


print("Loss function definida")

In [ ]:
# ============================================================
# LOOP DE ENTRENAMIENTO
# ============================================================

train_losses = []
val_losses = []
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(N_EPOCHS):
    # --- Train ---
    model.train()
    epoch_loss = 0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS} [train]")
    for images, texts in pbar:
        images = images.to(device)
        texts = texts.to(device)

        loss = clip_loss(model, images, texts)

        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping para estabilidad
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

    avg_train_loss = epoch_loss / n_batches
    train_losses.append(avg_train_loss)

    # --- Validation ---
    model.eval()
    val_loss = 0
    val_batches = 0

    with torch.no_grad():
        for images, texts in val_loader:
            images = images.to(device)
            texts = texts.to(device)
            loss = clip_loss(model, images, texts)
            val_loss += loss.item()
            val_batches += 1

    avg_val_loss = val_loss / val_batches
    val_losses.append(avg_val_loss)

    print(f"  Epoch {epoch+1}: train_loss={avg_train_loss:.4f}  val_loss={avg_val_loss:.4f}")

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "clip_pastis_v1_best.pt")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping en epoca {epoch+1} (sin mejora en {PATIENCE} epocas)")
            break

print(f"\nMejor val_loss: {best_val_loss:.4f}")

---
## 5. Resultados

In [ ]:
# ============================================================
# CURVAS DE LOSS
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(train_losses)+1), train_losses, "o-", label="Train loss")
ax.plot(range(1, len(val_losses)+1), val_losses, "s-", label="Val loss")
ax.set_xlabel("Epoca")
ax.set_ylabel("Loss (InfoNCE)")
ax.set_title("Curvas de entrenamiento CLIP — PASTIS-R")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Evaluacion — Retrieval y similitud

In [ ]:
# ============================================================
# RETRIEVAL METRICS sobre validacion
# ============================================================

# Cargar mejor checkpoint
model.load_state_dict(torch.load("clip_pastis_v1_best.pt", map_location=device, weights_only=True))
model.eval()

# Recolectar embeddings de todo el set de validacion
all_img_feats = []
all_txt_feats = []

with torch.no_grad():
    for images, texts in tqdm(val_loader, desc="Encoding val set"):
        images = images.to(device)
        texts = texts.to(device)

        img_f = model.encode_image(images)
        txt_f = model.encode_text(texts)

        all_img_feats.append(F.normalize(img_f, dim=-1).cpu())
        all_txt_feats.append(F.normalize(txt_f, dim=-1).cpu())

all_img_feats = torch.cat(all_img_feats)  # (N, D)
all_txt_feats = torch.cat(all_txt_feats)  # (N, D)

# Matriz de similitud completa
sim_matrix = all_img_feats @ all_txt_feats.t()  # (N, N)

# Recall@K: para cada imagen, ver si el caption correcto esta en el top-K
def recall_at_k(sim, k):
    n = sim.shape[0]
    topk = sim.topk(k, dim=1).indices  # (N, K)
    correct = torch.arange(n).unsqueeze(1).expand_as(topk)
    hits = (topk == correct).any(dim=1).float()
    return hits.mean().item()

r1  = recall_at_k(sim_matrix, 1)
r5  = recall_at_k(sim_matrix, 5)
r10 = recall_at_k(sim_matrix, 10)

print(f"Image-to-Text Retrieval (N={len(val_ids)}):")
print(f"  Recall@1:  {r1:.3f}")
print(f"  Recall@5:  {r5:.3f}")
print(f"  Recall@10: {r10:.3f}")

In [ ]:
# ============================================================
# HEATMAP de similitud (primeros 20 pares de validacion)
# ============================================================

N_SHOW = min(20, len(val_ids))
sim_sub = sim_matrix[:N_SHOW, :N_SHOW].numpy()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_sub, cmap="viridis", vmin=sim_sub.min(), vmax=sim_sub.max())
ax.set_xlabel("Texto (indice)")
ax.set_ylabel("Imagen (indice)")
ax.set_title(f"Matriz de similitud CLIP — primeros {N_SHOW} pares de validacion")
fig.colorbar(im, ax=ax, shrink=0.8)

# Marcar la diagonal (matches correctos)
for i in range(N_SHOW):
    ax.plot(i, i, "rx", markersize=8, markeredgewidth=2)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EJEMPLOS CUALITATIVOS
# ============================================================

# Para 5 imagenes de val, mostrar el caption real vs top-3 predichos
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

val_captions = []
for pid in val_ids:
    caption = (DATA_DIR / f"{pid}.txt").read_text(encoding="utf-8").strip()
    val_captions.append(caption)

for i in range(5):
    # Imagen
    img = Image.open(DATA_DIR / f"{val_ids[i]}.png")
    axes[i].imshow(img)
    axes[i].axis("off")

    # Top-3 textos mas similares segun CLIP
    sims = sim_matrix[i]
    top3 = sims.topk(3).indices.tolist()

    real_crops = val_captions[i].replace("A satellite image containing: ", "")
    pred_crops = [val_captions[j].replace("A satellite image containing: ", "") for j in top3]

    title = f"Patch {val_ids[i]}\n"
    title += f"Real: {real_crops[:40]}...\n" if len(real_crops) > 40 else f"Real: {real_crops}\n"
    match_str = "match" if top3[0] == i else f"pred #{top3[0]}"
    title += f"Top1: {match_str}"
    axes[i].set_title(title, fontsize=7)

plt.suptitle("Ejemplos: imagen → caption (real vs prediccion CLIP)", fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. Guardado del modelo

In [ ]:
# El mejor checkpoint ya se guardo durante entrenamiento como clip_pastis_v1_best.pt
# Guardar tambien el estado final
torch.save({
    "model_state_dict": model.state_dict(),
    "train_losses": train_losses,
    "val_losses": val_losses,
    "best_val_loss": best_val_loss,
    "config": {
        "model": "ViT-B-32",
        "pretrained": "laion2b_s34b_b79k",
        "lr": LR,
        "batch_size": BATCH_SIZE,
        "epochs_trained": len(train_losses),
        "train_size": len(train_ids),
        "val_size": len(val_ids),
    }
}, "clip_pastis_v1_final.pt")

print("Modelo guardado:")
print(f"  clip_pastis_v1_best.pt   (mejor val_loss={best_val_loss:.4f})")
print(f"  clip_pastis_v1_final.pt  (estado final + config + metricas)")